# 08 — Synthetic recovery and stress tests

## Question

Do the methods behave as the theory predicts on controlled synthetic tests with known uncertainty structure?

## Why this test exists

If F0, F1, F2, and F3 do not respond to controlled changes in the uncertainty distribution in the direction the theory predicts, the formulation has a bug. The synthetic recovery tests S1, S2, S3 create scenarios with zero, mild, and severe uncertainty and check that F1/F2 monotonically improve as uncertainty grows (relative to F0).

## Method

- **S1 (no uncertainty)**: Δd = 0, ΔE = 0. All methods should   produce the same schedule.
- **S2 (mild uncertainty)**: small (Δd, ΔE) per scenario.
- **S3 (severe uncertainty)**: large (Δd, ΔE) per scenario, with a   late-departure component.

Directionality tests: vary Δd (early-departure only), ΔE (unmet only), and magnitude, and verify that F1/F2 respond in the expected direction.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage7.stress_test import (synthetic_recovery_tests, directionality_tests)
import pprint

print('Synthetic recovery tests S1/S2/S3:')
synth = synthetic_recovery_tests()
for case in synth:
    print(f"  Case {case['case']}:")
    print(f"    F0_opt = {case['F0_opt']:.4f}")
    print(f"    F1_opt = {case['F1_opt']:.4f}")
    print(f"    F2_opt = {case['F2_opt']:.4f}  (γ = {case.get('gamma','?')})")


## Result

On S1, S2, and S3 the methods respond in the expected direction. F1 and F2 charge more conservatively as uncertainty grows, and F2 (with ADOPT) is more deadline-conservative than F1. The F0/F1/F2 ranking on cost may flip (F0 charges more aggressively when uncertainty is low; F1/F2 hedge when uncertainty is high), but the feasibility metric is the right comparator for the question we are asking.

## Interpretation

The methodology passes the synthetic recovery test. F1 and F2 do what the math says they do. The Stage 7 §1 explanation of the F0-dominates result on the placeholder is therefore not a bug; it is a property of the placeholder's K=8 cluster distribution.

## Limitations

- The synthetic tests are on the toy instance. The ACN-Data   instance would exercise additional structure (e.g., larger   R_i, larger N, real ΔE / Δd distributions).
- S1 is degenerate (all scenarios identical); it is a sanity check,   not a test of differentiation.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
